
<table style="border: none; width: 100%; border-collapse: collapse;">
  <tr>
    <td style="border: none; text-align: center;">
      <img src="https://raw.githubusercontent.com/cristiandarioortegayubro/databricks-finance-lab/main/Imagenes/uda.jpg" width="90"/>
    </td>
    <td style="border: none">
      <h1>Universidad del Aconcagua</h1>
      <h2>Facultad de Ciencias Económicas y Jurídicas</h2>
    </td>
  </tr>
</table>

---

# Databricks Finance Lab
## Analítica Financiera Agéntica

### Módulo 05: Analítica Agéntica con IA
### Notebook 5.2: Análisis de Datos Financieros con IA
### 📊 **CASOS PRÁCTICOS CON IA**

---

# 5.2 - Analisis de Datos Financieros con IA

## Objetivo
Aplicar IA para analizar datos financieros reales y generar insights.

## Que Aprenderemos
* Analisis de series temporales con IA
* Deteccion de anomalias
* Predicciones basicas
* Generacion de reportes automaticos
* Optimizacion de portafolios con IA

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_style('whitegrid')

# Simular datos de 5 acciones durante 2 anos
np.random.seed(42)
n_dias = 504  # 2 anos aprox
fecha_inicio = datetime(2022, 1, 1)

fechas = pd.date_range(fecha_inicio, periods=n_dias, freq='D')

# Simular precios de acciones
precios_iniciales = [100, 50, 150, 80, 120]
acciones = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']

precios_dict = {}
for i, accion in enumerate(acciones):
    retornos = np.random.normal(0.0005, 0.02, n_dias)
    precios = precios_iniciales[i] * np.exp(np.cumsum(retornos))
    precios_dict[accion] = precios

df_precios = pd.DataFrame(precios_dict, index=fechas)

print("Dataset de precios creado")
print(f"Periodo: {df_precios.index[0].date()} a {df_precios.index[-1].date()}")
print(f"Acciones: {', '.join(acciones)}")
print(f"\nPrimeros precios:")
print(df_precios.head())

## Caso 1: Analisis de Tendencias con IA

### Problema
Identificar cual accion ha tenido mejor desempeno en diferentes periodos.

In [0]:
# Graficar evolucion de precios normalizados
df_normalizado = df_precios / df_precios.iloc[0] * 100

plt.figure(figsize=(14, 7))
for col in df_normalizado.columns:
    plt.plot(df_normalizado.index, df_normalizado[col], label=col, linewidth=2)

plt.title('Evolucion de Precios (Base 100)', fontsize=14, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Precio Normalizado')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nConsulta con Genie:")
print('"Cual accion tuvo mejor retorno acumulado?"')
print('"Identifica periodos de alta volatilidad"')

## Caso 2: Deteccion de Anomalias

### Problema
Identificar dias con movimientos anormales en los precios.

In [0]:
# Calcular retornos diarios
df_retornos = df_precios.pct_change().dropna()

# Identificar retornos extremos (>3 desviaciones estandar)
for accion in acciones:
    media = df_retornos[accion].mean()
    std = df_retornos[accion].std()
    umbral_superior = media + 3 * std
    umbral_inferior = media - 3 * std
    
    anomalias = df_retornos[
        (df_retornos[accion] > umbral_superior) | 
        (df_retornos[accion] < umbral_inferior)
    ]
    
    if len(anomalias) > 0:
        print(f"\n{accion}:")
        print(f"  Anomalias detectadas: {len(anomalias)}")
        print(f"  Mayor caida: {anomalias[accion].min()*100:.2f}%")
        print(f"  Mayor subida: {anomalias[accion].max()*100:.2f}%")

print("\n" + "="*60)
print("Consulta con Genie:")
print('"Grafica los retornos de AAPL con los outliers marcados"')
print('"Que dias tuvieron movimientos anomalos simultaneos?"')

## Caso 3: Analisis de Correlaciones

### Problema
Identificar acciones que se mueven juntas para optimizar diversificacion.

In [0]:
# Calcular matriz de correlacion
corr_matrix = df_retornos.corr()

# Visualizar con heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlacion de Retornos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nPares con mayor correlacion:")
corr_pairs = []
for i in range(len(acciones)):
    for j in range(i+1, len(acciones)):
        corr_pairs.append((
            acciones[i], 
            acciones[j], 
            corr_matrix.iloc[i, j]
        ))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for i, (a1, a2, corr) in enumerate(corr_pairs[:3]):
    print(f"{i+1}. {a1} - {a2}: {corr:.3f}")

print("\n" + "="*60)
print("Consulta con Genie:")
print('"Sugiere un portafolio de 3 acciones con baja correlacion"')
print('"Calcula el beneficio de diversificacion"')

## Caso 4: Optimizacion Automatica de Portafolio

### Problema
Encontrar los pesos optimos para maximizar Sharpe Ratio.

In [0]:
# Calcular metricas anualizadas
retorno_anual = df_retornos.mean() * 252
volatilidad_anual = df_retornos.std() * np.sqrt(252)
cov_matrix = df_retornos.cov() * 252

# Simular 5000 portafolios aleatorios
n_portafolios = 5000
resultados = np.zeros((3, n_portafolios))
pesos_portafolios = []

np.random.seed(42)
for i in range(n_portafolios):
    pesos = np.random.random(len(acciones))
    pesos /= pesos.sum()
    pesos_portafolios.append(pesos)
    
    ret_port = np.dot(pesos, retorno_anual)
    vol_port = np.sqrt(np.dot(pesos.T, np.dot(cov_matrix, pesos)))
    sharpe = ret_port / vol_port
    
    resultados[0, i] = ret_port
    resultados[1, i] = vol_port
    resultados[2, i] = sharpe

# Encontrar portafolio optimo
idx_max_sharpe = resultados[2].argmax()
mejor_retorno = resultados[0, idx_max_sharpe]
mejor_volatilidad = resultados[1, idx_max_sharpe]
mejor_sharpe = resultados[2, idx_max_sharpe]
mejores_pesos = pesos_portafolios[idx_max_sharpe]

print("PORTAFOLIO OPTIMO (Maximo Sharpe Ratio)")
print("="*60)
for i, accion in enumerate(acciones):
    print(f"{accion}: {mejores_pesos[i]*100:.2f}%")
print(f"\nRetorno esperado: {mejor_retorno*100:.2f}%")
print(f"Volatilidad: {mejor_volatilidad*100:.2f}%")
print(f"Sharpe Ratio: {mejor_sharpe:.3f}")

In [0]:
# Graficar frontera eficiente
plt.figure(figsize=(12, 8))

# Scatter de todos los portafolios
sc = plt.scatter(resultados[1]*100, resultados[0]*100, 
                 c=resultados[2], cmap='viridis', alpha=0.6, s=10)
plt.colorbar(sc, label='Sharpe Ratio')

# Marcar portafolio optimo
plt.scatter(mejor_volatilidad*100, mejor_retorno*100, 
            marker='*', color='red', s=500, edgecolors='black', 
            linewidths=2, label='Portafolio Optimo')

# Marcar acciones individuales
for i, accion in enumerate(acciones):
    plt.scatter(volatilidad_anual[accion]*100, retorno_anual[accion]*100,
                marker='o', s=200, edgecolors='black', linewidths=2,
                label=accion)

plt.xlabel('Volatilidad Anualizada (%)', fontsize=12)
plt.ylabel('Retorno Esperado Anualizado (%)', fontsize=12)
plt.title('Frontera Eficiente - Optimizacion con IA', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nConsulta con Genie:")
print('"Encuentra el portafolio de minima varianza"')
print('"Compara este portafolio con uno equiponderado"')

## Resumen

### Casos Analizados
1. **Tendencias**: Identificacion de mejor desempeno
2. **Anomalias**: Deteccion de movimientos extremos
3. **Correlaciones**: Analisis de diversificacion
4. **Optimizacion**: Busqueda de portafolio optimo

### Ejercicios con Genie

**Ejercicio 1**: Analisis Temporal
* Divide los datos en 4 trimestres
* Identifica cual trimestre tuvo mayor volatilidad
* Grafica el retorno acumulado por trimestre

**Ejercicio 2**: Riesgo vs Retorno
* Calcula el Coefficient of Variation de cada accion
* Identifica la accion con mejor retorno ajustado por riesgo
* Crea un ranking de acciones

**Ejercicio 3**: Backtesting
* Crea una estrategia simple (ej: comprar la mejor accion del mes anterior)
* Calcula el retorno de la estrategia
* Comparala con comprar y mantener

**Ejercicio 4**: Reporte Automatico
* Usa Genie para generar un reporte ejecutivo
* Incluye metricas clave, graficos y recomendaciones
* Exporta a formato presentable

### Proximos Pasos
* **Notebook 5.3**: Caso integrador - Portfolio inteligente completo